In [1]:
print('hi')

hi


In [2]:
from __future__ import annotations

from typing import Any

import gymnasium as gym
import numpy as np
from gymnasium import spaces

from snfs_traffic.rl.episode import EpisodeConfig, build_reset_info
from snfs_traffic.rl.multiagent_env import SnfsTrafficMultiAgentEnv
from snfs_traffic.rl.rewards import RewardConfig
from snfs_traffic.scenarios import VehicleMix
from snfs_traffic.simulator import ScenarioConfig, TrafficSimulator


class PriorityTrafficMultiAgentEnv(SnfsTrafficMultiAgentEnv):
    """
    Experimental multi-agent env:

    - 4-lane periodic ring road;
    - road length = 1000 cells;
    - HDV vehicles: uncontrolled Rev S-NFS;
    - priority vehicles: uncontrolled Rev S-NFS, tracked separately;
    - RL agents: controlled vehicles, one shared policy;
    - observation: Dict({"near_space": Box(shape=(4, M))});
    - near_space[lane, col] = 0 if empty, else velocity + 1.
    """

    def __init__(
        self,
        *,
        density_range: tuple[float, float] = (0.08, 0.20),
        priority_count_range: tuple[int, int] = (3, 8),
        agent_fraction_range: tuple[float, float] = (0.05, 0.15),
        near_m: int = 101,
        backend: str = "reference",
        episode_config: EpisodeConfig | None = None,
        reward_config: RewardConfig | None = None,
        seed: int | None = None,
        scenario_seed: int | None = None,
    ) -> None:
        self.density_min, self.density_max = self._validate_float_range(
            "density_range",
            density_range,
            min_value=0.0,
            max_value=1.0,
        )
        self.priority_min, self.priority_max = self._validate_int_range(
            "priority_count_range",
            priority_count_range,
            min_value=0,
        )
        self.agent_fraction_min, self.agent_fraction_max = self._validate_float_range(
            "agent_fraction_range",
            agent_fraction_range,
            min_value=0.0,
            max_value=1.0,
        )

        if isinstance(near_m, bool) or not isinstance(near_m, int) or near_m < 1:
            raise ValueError("near_m must be a positive int")
        if near_m % 2 == 0:
            raise ValueError("near_m should be odd so the ego vehicle has a center column")

        self.near_m = near_m
        self._near_center = near_m // 2
        self._near_offsets = np.arange(near_m, dtype=np.int64) - self._near_center

        self._priority_vehicle_ids: frozenset[int] = frozenset()
        self._agent_vehicle_ids: frozenset[int] = frozenset()
        self._hdv_vehicle_ids: frozenset[int] = frozenset()
        self._last_reset_sample: dict[str, float | int] = {}

        # num_controlled=1 is only a constructor placeholder.
        # Actual number of RL agents is sampled in _select_controlled().
        super().__init__(
            num_lanes=4,
            road_length=1000,
            density=self.density_min,
            num_controlled=1,
            backend=backend,
            episode_config=episode_config,
            reward_config=reward_config,
            seed=seed,
            scenario_seed=scenario_seed,
        )

        vmax_default = getattr(self.params, "vmax_default", 0)
        vmax_controlled = getattr(self.params, "vmax_controlled", vmax_default)
        obs_high = float(max(vmax_default, vmax_controlled) + 1)

        self.single_agent_observation_space = spaces.Dict(
            {
                "near_space": spaces.Box(
                    low=0.0,
                    high=obs_high,
                    shape=(self.params.num_lanes, self.near_m),
                    dtype=np.float32,
                )
            }
        )
        self.observation_space = self.single_agent_observation_space

    def reset(self, *, seed: int | None = None, options: dict[str, Any] | None = None):
        """
        Override reset only because density is sampled per episode.

        The rest mirrors SnfsTrafficMultiAgentEnv.reset():
        - generate scenario state;
        - select controlled RL-agent vehicles;
        - reset simulator with marked controlled vehicles;
        - return observation/info dicts keyed by agent id.
        """
        env_seed = seed if seed is not None else self._initial_seed
        gym.Env.reset(self, seed=env_seed)

        options = options or {}

        scenario_seed = options.get("scenario_seed", self._scenario_seed)
        rng_seed = options.get("rng_seed", seed if seed is not None else self._initial_seed)

        if scenario_seed is None:
            scenario_seed = int(self.np_random.integers(0, np.iinfo(np.int32).max))
        if rng_seed is None:
            rng_seed = int(self.np_random.integers(0, np.iinfo(np.int32).max))

        density = float(
            options.get(
                "density",
                self.np_random.uniform(self.density_min, self.density_max),
            )
        )
        if not (0.0 <= density <= 1.0):
            raise ValueError(f"density must be in [0, 1], got {density}")

        self._last_reset_sample = {
            "density": density,
            "scenario_seed": int(scenario_seed),
            "rng_seed": int(rng_seed),
        }

        self._sim = TrafficSimulator(
            params=self.params,
            backend=self._backend,
            rng_seed=int(rng_seed),
            scenario=ScenarioConfig(
                density=density,
                seed=int(scenario_seed),
                vehicle_mix=VehicleMix(),
            ),
            require_all_controlled_actions=True,
        )

        state = self._sim.reset(seed=int(scenario_seed), rng_seed=int(rng_seed))
        state = self._select_controlled(state)
        state = self._sim.reset(state=state, rng_seed=int(rng_seed))

        self._step_index = 0
        self._done = False

        observations = self._build_observations()

        infos: dict[str, dict[str, Any]] = {}
        for agent_id in self.agents:
            vehicle_id = self._agent_to_vehicle_id[agent_id]
            info = build_reset_info(
                state,
                controlled_vehicle_id=vehicle_id,
                step_index=self._step_index,
                backend_name=self._sim.backend_name,
                params=self.params,
            )
            info["agent_id"] = agent_id
            info["sampled_density"] = density
            info["num_priority_vehicles"] = len(self._priority_vehicle_ids)
            info["num_rl_agents"] = len(self._agent_vehicle_ids)
            info["num_hdv_vehicles"] = len(self._hdv_vehicle_ids)
            infos[agent_id] = info

        return observations, infos

    def _select_controlled(self, state):
        """
        Select vehicle roles at reset.

        Important:
        - only RL agents become state.controlled=True;
        - priority vehicles remain uncontrolled and still use Rev S-NFS;
        - HDV vehicles remain uncontrolled and still use Rev S-NFS.
        """
        alive_ids = np.asarray(state.vehicle_id[state.alive], dtype=np.int64)
        if alive_ids.size < 2:
            raise ValueError("scenario must contain at least 2 alive vehicles")

        requested_priority_count = int(
            self.np_random.integers(self.priority_min, self.priority_max + 1)
        )

        # Keep at least one non-priority vehicle so at least one RL agent can exist.
        priority_count = min(requested_priority_count, alive_ids.size - 1)

        if priority_count > 0:
            priority_ids = set(
                int(v)
                for v in self.np_random.choice(
                    alive_ids,
                    size=priority_count,
                    replace=False,
                )
            )
        else:
            priority_ids = set()

        non_priority_ids = np.asarray(
            [int(v) for v in alive_ids if int(v) not in priority_ids],
            dtype=np.int64,
        )
        if non_priority_ids.size < 1:
            raise ValueError("scenario must contain at least one non-priority vehicle")

        agent_fraction = float(
            self.np_random.uniform(self.agent_fraction_min, self.agent_fraction_max)
        )
        agent_count = int(round(agent_fraction * non_priority_ids.size))
        agent_count = max(1, agent_count)
        agent_count = min(agent_count, non_priority_ids.size)

        agent_ids = set(
            int(v)
            for v in self.np_random.choice(
                non_priority_ids,
                size=agent_count,
                replace=False,
            )
        )

        hdv_ids = {
            int(v)
            for v in alive_ids
            if int(v) not in priority_ids and int(v) not in agent_ids
        }

        state = state.copy()
        state.controlled[:] = False

        index_by_vehicle_id = {
            int(vehicle_id): i
            for i, vehicle_id in enumerate(state.vehicle_id)
        }

        for vehicle_id in agent_ids:
            idx = index_by_vehicle_id[vehicle_id]
            if not bool(state.alive[idx]):
                raise ValueError(f"selected RL agent vehicle {vehicle_id} is not alive")
            state.controlled[idx] = True

        selected = tuple(sorted(agent_ids))

        self._priority_vehicle_ids = frozenset(priority_ids)
        self._agent_vehicle_ids = frozenset(agent_ids)
        self._hdv_vehicle_ids = frozenset(hdv_ids)

        self._controlled_vehicle_ids = selected
        self._set_active_agents(selected)

        self._last_reset_sample["requested_priority_count"] = requested_priority_count
        self._last_reset_sample["actual_priority_count"] = len(priority_ids)
        self._last_reset_sample["agent_fraction"] = agent_fraction
        self._last_reset_sample["agent_count"] = len(agent_ids)
        self._last_reset_sample["hdv_count"] = len(hdv_ids)

        return state

    def _build_observations(self) -> dict[str, dict[str, np.ndarray]]:
        """
        Fast NumPy near_space builder.

        Complexity:
        - build dense occupancy once: O(num_vehicles);
        - gather all agent windows: O(num_agents * num_lanes * M);
        - no per-agent scan over all vehicles.
        """
        state = self._sim.state

        occ = np.zeros(
            (self.params.num_lanes, self.params.road_length),
            dtype=np.float32,
        )

        alive_idx = np.flatnonzero(state.alive)

        # Fast path for the current important case: vehicle length == 1.
        # Fallback below still handles length > 1.
        if alive_idx.size > 0 and np.all(state.length[alive_idx] == 1):
            lanes = state.lane[alive_idx].astype(np.int64, copy=False)
            positions = state.pos[alive_idx].astype(np.int64, copy=False)
            values = state.vel[alive_idx].astype(np.float32, copy=False) + 1.0
            occ[lanes, positions] = values
        else:
            for i in alive_idx:
                lane = int(state.lane[i])
                pos = int(state.pos[i])
                vel = int(state.vel[i])
                length = int(state.length[i])

                for d in range(length):
                    cell = (pos + d) % self.params.road_length
                    occ[lane, cell] = float(vel + 1)

        if not self.agents:
            return {}

        index_by_vehicle_id = {
            int(vehicle_id): i
            for i, vehicle_id in enumerate(state.vehicle_id)
        }

        agent_indices = np.asarray(
            [
                index_by_vehicle_id[self._agent_to_vehicle_id[agent_id]]
                for agent_id in self.agents
            ],
            dtype=np.int64,
        )

        ego_positions = state.pos[agent_indices].astype(np.int64, copy=False)
        cols = (ego_positions[:, None] + self._near_offsets[None, :]) % self.params.road_length

        # occ[:, cols] shape is [num_lanes, num_agents, M].
        # We want [num_agents, num_lanes, M].
        near_batch = np.transpose(occ[:, cols], (1, 0, 2)).astype(np.float32, copy=False)

        return {
            agent_id: {"near_space": near_batch[i]}
            for i, agent_id in enumerate(self.agents)
        }

    def _compute_agent_reward(
        self,
        prev_state,
        next_state,
        *,
        agent_id: str,
        controlled_vehicle_id: int,
        action: object,
        action_applied: bool | None,
    ) -> tuple[float, dict[str, float]]:
        """
        Simple experimental reward.

        Components:
        - reward for agent speed;
        - penalty for being stopped;
        - small penalty for requesting lane change;
        - penalty if the agent is likely blocking a priority vehicle behind it.
        """
        idx = self._vehicle_index(next_state, controlled_vehicle_id)
        if idx is None or not bool(next_state.alive[idx]):
            return -1.0, {
                "speed_reward": 0.0,
                "stopped_penalty": -1.0,
                "lane_change_penalty": 0.0,
                "blocked_action_penalty": 0.0,
                "priority_block_penalty": 0.0,
                "total": -1.0,
            }

        vmax_default = max(int(getattr(self.params, "vmax_default", 1)), 1)
        vmax_controlled = max(int(getattr(self.params, "vmax_controlled", vmax_default)), 1)
        vmax = max(vmax_default, vmax_controlled)

        vel = float(next_state.vel[idx])
        speed_reward = vel / float(vmax)
        stopped_penalty = -0.25 if vel <= 0.0 else 0.0

        lane_delta = self._extract_lane_delta(action, controlled_vehicle_id)
        lane_change_penalty = -0.03 if lane_delta != 0 else 0.0
        blocked_action_penalty = -0.10 if lane_delta != 0 and action_applied is False else 0.0

        priority_block_penalty = self._priority_block_penalty(next_state, idx)

        total = (
            speed_reward
            + stopped_penalty
            + lane_change_penalty
            + blocked_action_penalty
            + priority_block_penalty
        )

        return float(total), {
            "speed_reward": float(speed_reward),
            "stopped_penalty": float(stopped_penalty),
            "lane_change_penalty": float(lane_change_penalty),
            "blocked_action_penalty": float(blocked_action_penalty),
            "priority_block_penalty": float(priority_block_penalty),
            "total": float(total),
        }

    def _priority_block_penalty(self, state, agent_idx: int) -> float:
        """
        Penalize an RL agent if a priority vehicle is close behind in the same lane.

        This is intentionally simple. Later this can be replaced with a better
        delay-based priority metric.
        """
        if not self._priority_vehicle_ids:
            return 0.0

        agent_lane = int(state.lane[agent_idx])
        agent_pos = int(state.pos[agent_idx])

        lookback = 20
        penalty = 0.0
        vmax_default = max(int(getattr(self.params, "vmax_default", 1)), 1)

        for i in range(state.n_vehicles):
            if not bool(state.alive[i]):
                continue

            vehicle_id = int(state.vehicle_id[i])
            if vehicle_id not in self._priority_vehicle_ids:
                continue

            if int(state.lane[i]) != agent_lane:
                continue

            priority_pos = int(state.pos[i])
            dist_behind = (agent_pos - priority_pos) % self.params.road_length

            if 0 < dist_behind <= lookback:
                priority_vel = float(state.vel[i])
                priority_slowdown = max(0.0, 1.0 - priority_vel / float(vmax_default))
                proximity_weight = 1.0 - dist_behind / float(lookback + 1)
                penalty -= 0.20 * proximity_weight * priority_slowdown

        return float(penalty)

    @staticmethod
    def _extract_lane_delta(action: object, controlled_vehicle_id: int) -> int:
        if isinstance(action, dict):
            return int(action.get(controlled_vehicle_id, 0))
        return int(action)

    @staticmethod
    def _vehicle_index(state, vehicle_id: int) -> int | None:
        matches = np.flatnonzero(state.vehicle_id == int(vehicle_id))
        if matches.size == 0:
            return None
        return int(matches[0])

    @staticmethod
    def _validate_float_range(
        name: str,
        value: tuple[float, float],
        *,
        min_value: float,
        max_value: float,
    ) -> tuple[float, float]:
        if len(value) != 2:
            raise ValueError(f"{name} must be a pair")
        lo, hi = float(value[0]), float(value[1])
        if not (min_value <= lo <= hi <= max_value):
            raise ValueError(f"{name} must satisfy {min_value} <= min <= max <= {max_value}")
        return lo, hi

    @staticmethod
    def _validate_int_range(
        name: str,
        value: tuple[int, int],
        *,
        min_value: int,
    ) -> tuple[int, int]:
        if len(value) != 2:
            raise ValueError(f"{name} must be a pair")
        lo, hi = value
        if isinstance(lo, bool) or isinstance(hi, bool):
            raise ValueError(f"{name} values must be ints")
        if not isinstance(lo, int) or not isinstance(hi, int):
            raise ValueError(f"{name} values must be ints")
        if not (min_value <= lo <= hi):
            raise ValueError(f"{name} must satisfy {min_value} <= min <= max")
        return lo, hi


def shared_policy(agent_obs: dict[str, np.ndarray]) -> int:
    """
    Placeholder shared policy.

    All agents use this same function. For now it always keeps lane.
    Later this can be replaced by a neural policy.
    """
    near_space = agent_obs["near_space"]

    # Example of reading the ego cell, though this policy does not use it yet.
    _ego_column = near_space.shape[1] // 2

    return 0




In [3]:
env = PriorityTrafficMultiAgentEnv(
        backend="reference",
        density_range=(0.08, 0.20),
        priority_count_range=(3, 8),
        agent_fraction_range=(0.05, 0.10),
        near_m=101,
        episode_config=EpisodeConfig(max_steps=20),
        seed=123,
    )

obs, infos = env.reset(seed=123)

first_info = next(iter(infos.values()))
print("reset:")
print(f"  agents:   {len(env.agents)}")
print(f"  priority: {first_info['num_priority_vehicles']}")
print(f"  hdv:      {first_info['num_hdv_vehicles']}")
print(f"  density:  {first_info['sampled_density']:.4f}")
print(f"  obs keys: {list(next(iter(obs.values())).keys())}")
print(f"  near_space shape: {next(iter(obs.values()))['near_space'].shape}")

for step_idx in range(20):
    actions = {
        agent_id: shared_policy(obs[agent_id])
        for agent_id in env.agents
    }

    obs, rewards, terminateds, truncateds, infos = env.step(actions)

    mean_reward = float(np.mean(list(rewards.values()))) if rewards else 0.0
    print(
        f"step={step_idx + 1:02d} "
        f"agents={len(env.agents):3d} "
        f"mean_reward={mean_reward:+.4f} "
        f"done={terminateds['__all__'] or truncateds['__all__']}"
    )

    if terminateds["__all__"] or truncateds["__all__"]:
        break



reset:
  agents:   32
  priority: 7
  hdv:      306
  density:  0.0865
  obs keys: ['near_space']
  near_space shape: (4, 101)
step=01 agents= 32 mean_reward=-0.1737 done=False
step=02 agents= 32 mean_reward=-0.1086 done=False
step=03 agents= 32 mean_reward=-0.0304 done=False
step=04 agents= 32 mean_reward=+0.0402 done=False
step=05 agents= 32 mean_reward=+0.0980 done=False
step=06 agents= 32 mean_reward=+0.1375 done=False
step=07 agents= 32 mean_reward=+0.1973 done=False
step=08 agents= 32 mean_reward=+0.2876 done=False
step=09 agents= 32 mean_reward=+0.3617 done=False
step=10 agents= 32 mean_reward=+0.4254 done=False
step=11 agents= 32 mean_reward=+0.4796 done=False
step=12 agents= 32 mean_reward=+0.4890 done=False
step=13 agents= 32 mean_reward=+0.5117 done=False
step=14 agents= 32 mean_reward=+0.5352 done=False
step=15 agents= 32 mean_reward=+0.5614 done=False
step=16 agents= 32 mean_reward=+0.5789 done=False
step=17 agents= 32 mean_reward=+0.5913 done=False
step=18 agents= 32 mean